In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from scripts.plotting import *
from scripts.TPS import *
from scripts.denoising import *

In [ ]:
import scanpy as sc
import scvelo as scv

bdata = sc.read_h5ad("./data/pancreas/pancreas_inferred_velocity.h5ad")
scv.pl.velocity_embedding_stream(bdata, basis="umap", color="clusters", density=1.5, arrow_size=0.1)

In [ ]:
bdata

In [ ]:
import numpy as np
import os

# Ensure the directory exists
save_dir = "data/RNA_velocity/pancreas_data"
os.makedirs(save_dir, exist_ok=True)

# Save the matrices
np.save(os.path.join(save_dir, "expression.npy"), bdata.layers["Ms"])
np.save(os.path.join(save_dir, "expression_umap.npy"), bdata.obsm["X_umap"])
np.save(os.path.join(save_dir, "velocity.npy"), bdata.layers["velocity"])

clusters = bdata.obs['clusters'].to_numpy()
np.save(os.path.join(save_dir, "clusters.npy"), clusters)

In [ ]:
from scripts.plotting import *
from scripts.TPS import *
from scripts.perturbation_distance import PerturbDistanceSolver

def scale_columns(X):
    return X / np.std(X, axis=0, keepdims=True)

X_2d = bdata.obsm["X_umap"]
X = bdata.layers["Ms"]
X = scale_columns(X)

Y = bdata.layers["velocity"]
Y = scale_columns(Y)

In [ ]:
r = 30
X_hat, Y_hat, V_hat = subspace_denoised(X, Y, r, alpha=0.5, method="sum")

In [ ]:
# %time solver = PerturbDistanceSolver(X_hat, Y_hat)
%time solver = PerturbDistanceSolver(X_hat, Y_hat)

In [ ]:
%time dist_mat = solver.pairwise_perturb_distance()

In [ ]:
%time t_dist = solver.pairwise_time_constrained_L2_distance()

In [ ]:
import umap.umap_ as umap

reducer = umap.UMAP(
    n_neighbors=30,
    n_components=2,
    metric="precomputed",
    random_state=1,
    min_dist=0.3
)
coords_umap = reducer.fit_transform(dist_mat)

cluster_labels = bdata.obs['clusters'].astype(str).values  # Ensure string type for mapping
cluster_colors = bdata.uns['clusters_colors']  # List of colors indexed by cluster order
unique_clusters = np.unique(cluster_labels)
cluster_to_color = {cluster: color for cluster, color in zip(unique_clusters, cluster_colors)}
cell_colors = [cluster_to_color[cluster] for cluster in cluster_labels]

plot_2d(-coords_umap, cell_colors, point_size=30)

In [ ]:
coords_umap = reducer.fit_transform(t_dist)

cluster_labels = bdata.obs['clusters'].astype(str).values  # Ensure string type for mapping
cluster_colors = bdata.uns['clusters_colors']  # List of colors indexed by cluster order
unique_clusters = np.unique(cluster_labels)
cluster_to_color = {cluster: color for cluster, color in zip(unique_clusters, cluster_colors)}
cell_colors = [cluster_to_color[cluster] for cluster in cluster_labels]

plot_2d(coords_umap, cell_colors, point_size=30)

In [ ]:
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
coords_mds = mds.fit_transform(t_dist)

plot_2d(coords_mds, cell_colors, point_size=30)

In [ ]:
solver = PerturbDistanceSolver(X, Y)
%time dist_mat = solver.pairwise_perturb_distance(alpha=0)
coords_umap = reducer.fit_transform(dist_mat)

cluster_labels = bdata.obs['clusters'].astype(str).values  # Ensure string type for mapping
cluster_colors = bdata.uns['clusters_colors']  # List of colors indexed by cluster order
unique_clusters = np.unique(cluster_labels)
cluster_to_color = {cluster: color for cluster, color in zip(unique_clusters, cluster_colors)}
cell_colors = [cluster_to_color[cluster] for cluster in cluster_labels]

plot_2d(coords_umap, cell_colors, point_size=30)

In [ ]:
from scripts.optimize_embedding import *

optimizer = EmbeddingOptimizer(X, Y, alpha=0.1, lam=1, method="umap", dof_target=50)
optimizer.run_full_pipeline()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
import umap
from scipy.spatial.distance import pdist, squareform

np.random.seed(42)
# 1. Sample time points
n_points = 1000
t = np.random.uniform(0, 1, size=n_points)

# 2. Compute pairwise distance matrix
dist_mat = squareform(pdist(t[:, None], metric='euclidean'))

# 3. MDS embedding
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
X_mds = mds.fit_transform(dist_mat)

# 4. UMAP embedding
umap_reducer = umap.UMAP(n_neighbors=30, n_components=2, metric="precomputed", min_dist=0.6, random_state=42)
X_umap = umap_reducer.fit_transform(dist_mat)

# 5. Plot
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

sc1 = axs[0].scatter(X_mds[:, 0], X_mds[:, 1], c=t, cmap='viridis')
axs[0].set_title('MDS Embedding')
fig.colorbar(sc1, ax=axs[0])

sc2 = axs[1].scatter(X_umap[:, 0], X_umap[:, 1], c=t, cmap='viridis')
axs[1].set_title('UMAP Embedding')
fig.colorbar(sc2, ax=axs[1])

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr
# 3. MDS 1D embedding
mds = MDS(n_components=1, dissimilarity="precomputed", random_state=42)
X_mds = mds.fit_transform(dist_mat).flatten()

# 4. UMAP 1D embedding
umap_reducer = umap.UMAP(n_neighbors=30, min_dist=0.6, n_components=1, metric="precomputed", random_state=42)
X_umap = umap_reducer.fit_transform(dist_mat).flatten()

# 5. Compute Spearman correlations
corr_mds, _ = spearmanr(X_mds, t)
corr_umap, _ = spearmanr(X_umap, t)

print(f"Spearman correlation (MDS): {corr_mds:.3f}")
print(f"Spearman correlation (UMAP): {corr_umap:.3f}")

# 6. Optional: scatter plots
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

axs[0].scatter(t, X_mds, s=20)
axs[0].set_title(f'MDS 1D embedding\nSpearman: {corr_mds:.3f}')
axs[0].set_xlabel('True time t')
axs[0].set_ylabel('Embedded coordinate')

axs[1].scatter(t, X_umap, s=20)
axs[1].set_title(f'UMAP 1D embedding\nSpearman: {corr_umap:.3f}')
axs[1].set_xlabel('True time t')

plt.tight_layout()
plt.show()


In [ ]:
np.random.seed(42)

# 1. Sample time points
n_points = 1000
t = np.random.uniform(0, 1, size=n_points)

# 2. Create 2D parabola curve: (x, y) = (t, t^2)
curve_points = np.stack([t, t**2], axis=1)

# 3. Compute pairwise distance matrix in 2D
dist_mat = squareform(pdist(curve_points, metric='euclidean'))

# 4. MDS embedding
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
X_mds = mds.fit_transform(dist_mat)

# 5. UMAP embedding
umap_reducer = umap.UMAP(n_neighbors=30, n_components=2, metric="precomputed", min_dist=0.6, random_state=42)
X_umap = umap_reducer.fit_transform(dist_mat)

# 6. Plot
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

sc1 = axs[0].scatter(X_mds[:, 0], X_mds[:, 1], c=t, cmap='viridis')
axs[0].set_title('MDS Embedding')
fig.colorbar(sc1, ax=axs[0])

sc2 = axs[1].scatter(X_umap[:, 0], X_umap[:, 1], c=t, cmap='viridis')
axs[1].set_title('UMAP Embedding')
fig.colorbar(sc2, ax=axs[1])

plt.tight_layout()
plt.show()


In [ ]:
mds = MDS(n_components=1, dissimilarity="precomputed", random_state=42)
X_mds = mds.fit_transform(dist_mat).flatten()

# 4. UMAP 1D embedding
umap_reducer = umap.UMAP(n_neighbors=30, min_dist=0.6, n_components=1, metric="precomputed", random_state=42)
X_umap = umap_reducer.fit_transform(dist_mat).flatten()

# 5. Compute Spearman correlations
corr_mds, _ = spearmanr(X_mds, t)
corr_umap, _ = spearmanr(X_umap, t)

print(f"Spearman correlation (MDS): {corr_mds:.3f}")
print(f"Spearman correlation (UMAP): {corr_umap:.3f}")

# 6. Optional: scatter plots
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

axs[0].scatter(t, X_mds, s=20)
axs[0].set_title(f'MDS 1D embedding\nSpearman: {corr_mds:.3f}')
axs[0].set_xlabel('True time t')
axs[0].set_ylabel('Embedded coordinate')

axs[1].scatter(t, X_umap, s=20)
axs[1].set_title(f'UMAP 1D embedding\nSpearman: {corr_umap:.3f}')
axs[1].set_xlabel('True time t')

plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# 1. Sample t from a symmetric range around 0
n_points = 1000
t = np.random.uniform(-1, 1, size=n_points)

# 2. Construct branching structure
points = []
colors = []

for ti in t:
    if ti < 0:
        points.append([ti, 0])
        colors.append(ti)
    else:
        # Two branches: upper and lower
        points.append([ti, ti])   # upper branch
        points.append([ti, -ti])  # lower branch
        colors.append(ti)
        colors.append(ti)

points = np.array(points)
colors = np.array(colors)

# 3. Compute pairwise distances
dist_mat = squareform(pdist(points, metric='euclidean'))

# 4. Embedding
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
X_mds = mds.fit_transform(dist_mat)

umap_reducer = umap.UMAP(n_neighbors=30, n_components=2, metric="precomputed", min_dist=0.6, random_state=42)
X_umap = umap_reducer.fit_transform(dist_mat)

# 5. Plotting
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

sc1 = axs[0].scatter(X_mds[:, 0], X_mds[:, 1], c=colors, cmap='coolwarm', s=10)
axs[0].set_title("MDS Embedding")
fig.colorbar(sc1, ax=axs[0])

sc2 = axs[1].scatter(X_umap[:, 0], X_umap[:, 1], c=colors, cmap='coolwarm', s=10)
axs[1].set_title("UMAP Embedding")
fig.colorbar(sc2, ax=axs[1])

plt.tight_layout()
plt.show()

In [ ]:
mds = MDS(n_components=1, dissimilarity="precomputed", random_state=42)
X_mds = mds.fit_transform(dist_mat).flatten()

# 4. UMAP 1D embedding
umap_reducer = umap.UMAP(n_neighbors=30, min_dist=0.6, n_components=1, metric="precomputed", random_state=42)
X_umap = umap_reducer.fit_transform(dist_mat).flatten()

# 5. Compute Spearman correlations
corr_mds, _ = spearmanr(X_mds, points[:,0])
corr_umap, _ = spearmanr(X_umap, points[:,0])

print(f"Spearman correlation (MDS): {corr_mds:.3f}")
print(f"Spearman correlation (UMAP): {corr_umap:.3f}")

# 6. Optional: scatter plots
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

axs[0].scatter(points[:,0], X_mds, s=20)
axs[0].set_title(f'MDS 1D embedding\nSpearman: {corr_mds:.3f}')
axs[0].set_xlabel('True time t')
axs[0].set_ylabel('Embedded coordinate')

axs[1].scatter(points[:,0], X_umap, s=20)
axs[1].set_title(f'UMAP 1D embedding\nSpearman: {corr_umap:.3f}')
axs[1].set_xlabel('True time t')

plt.tight_layout()
plt.show()